# KG Phase C Continuation

Embeds current repo code, copies previous A/B outputs from attached kernel output, and runs Phase C only from Phase B `best_composite`.

In [ ]:
import os, shutil, subprocess, glob, json, tarfile, sys, base64, io
from datetime import datetime

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
RUN_ID = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
CODE_ARCHIVE_B64 = """..."""
PREVIOUS_KERNEL_REF = "kgaero/rl-gspo-abc-current-selfcontained"

def write_progress(message):
    timestamp = datetime.utcnow().isoformat()
    print(f"{timestamp} {message}", flush=True)

write_progress(f"Phase C continuation notebook start (run_id={RUN_ID})")
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)
archive_bytes = base64.b64decode(CODE_ARCHIVE_B64.encode('ascii'))
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode='r:gz') as tf:
    tf.extractall(WORKDIR)
write_progress("Extracted embedded current repo code")
print("Workdir files:", sorted(os.listdir(WORKDIR)))


In [ ]:

def find_phase_b_bundle_source():
    # Kaggle may expose a single uploaded .tar.gz either as the archive file or as extracted contents.
    extracted = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.endswith("outputs_staged/phase_b/aliases") and "best_composite.json" in files:
            extracted.append(os.path.dirname(os.path.dirname(root)))
    if extracted:
        extracted = sorted(set(extracted))
        if len(extracted) > 1:
            write_progress(f"Multiple extracted Phase B bundle candidates found: {extracted}")
        return ("dir", extracted[0])

    tar_matches = glob.glob("/kaggle/input/**/kg_phase_b_best_composite_minimal.tar.gz", recursive=True)
    if tar_matches:
        tar_matches = sorted(tar_matches)
        if len(tar_matches) > 1:
            write_progress(f"Multiple Phase B tar candidates found: {tar_matches}")
        return ("tar", tar_matches[0])

    raise FileNotFoundError(
        "Could not find Phase B best_composite bundle under /kaggle/input. "
        f"Visible inputs: {glob.glob('/kaggle/input/*')}"
    )

source_kind, source_path = find_phase_b_bundle_source()
target_outputs = os.path.join(WORKDIR, "outputs_staged")
if os.path.exists(target_outputs):
    shutil.rmtree(target_outputs)

if source_kind == "dir":
    shutil.copytree(source_path, target_outputs)
    write_progress(f"Copied extracted Phase B outputs_staged from {source_path} to {target_outputs}")
else:
    with tarfile.open(source_path, "r:gz") as tf:
        tf.extractall(WORKDIR)
    write_progress(f"Extracted Phase B best_composite tar from {source_path} into {WORKDIR}")

with open(os.path.join(target_outputs, "phase_b", "aliases", "best_composite.json"), "r", encoding="utf-8") as handle:
    print("Phase B best_composite alias:", json.dumps(json.load(handle), indent=2)[:2000])


In [ ]:
write_progress("Installing uv and creating venv")
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "numpy==1.26.4", "scipy==1.15.3", "scikit-learn==1.6.1"],
    [
        VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir",
        "pillow", "packaging", "safetensors", "torchvision", "bitsandbytes", "xformers",
        "huggingface_hub>=0.34.0", "datasets==4.3.0", "transformers==4.56.2",
        "trl==0.22.2", "unsloth", "modelscope",
    ],
    [
        VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "vllm==0.10.2",
        "--extra-index-url", "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=WORKDIR)
write_progress("Finished venv package installs")


In [ ]:
compat_script = """
import numpy, scipy, sklearn, torch, transformers, trl, unsloth, vllm
print({
    'numpy': numpy.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
    'torch': torch.__version__, 'transformers': transformers.__version__, 'trl': trl.__version__, 'vllm': vllm.__version__,
})
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)
write_progress("Compatibility import check completed")


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
TRAIN_SPLIT = "test"
EVAL_SPLIT = "testmini"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4

def resolve_model_path():
    candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)
    matches = []
    for config_path in candidates:
        root = os.path.dirname(config_path)
        if os.path.exists(os.path.join(root, "model.safetensors")):
            matches.append(root)
    if not matches:
        raise FileNotFoundError("Could not find model config.json + model.safetensors under /kaggle/input.")
    if len(matches) > 1:
        write_progress(f"Multiple model candidates found: {matches}")
    return matches[0]

MODEL_PATH = resolve_model_path()
env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "0"
env["TRANSFORMERS_OFFLINE"] = "0"
env["HF_HUB_OFFLINE"] = "0"
env["HF_DATASETS_OFFLINE"] = "0"
write_progress(f"Resolved model path: {MODEL_PATH}")

cmd = [
    VENV_PYTHON,
    "rl_gspo_qwen2_5vlm_test3.py",
    "--phase", "phase_c",
    "--hardware-profile", HARDWARE_PROFILE,
    "--train-split", TRAIN_SPLIT,
    "--eval-split", EVAL_SPLIT,
    "--base-model-path", MODEL_PATH,
    "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    "--resume", "best_composite",
]
phase_output_dir = os.path.join(WORKDIR, "outputs_staged", "phase_c")
os.makedirs(phase_output_dir, exist_ok=True)
train_log_path = os.path.join(phase_output_dir, "train_subprocess.log")
write_progress("Starting phase_c continuation from copied Phase B best_composite")
print("Running:", " ".join(cmd), flush=True)
print("Subprocess log:", train_log_path, flush=True)
with open(train_log_path, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log_file.write(line)
        log_file.flush()
        print(line, end="", flush=True)
    return_code = process.wait()
if return_code != 0:
    write_progress(f"Phase C failed with return code {return_code}")
    print(f"Phase C failed with return code {return_code}. Last 200 log lines:", flush=True)
    with open(train_log_path, "r", encoding="utf-8") as log_file:
        print("".join(log_file.readlines()[-200:]), flush=True)
    raise subprocess.CalledProcessError(return_code, cmd)
write_progress("Phase C completed successfully")
